# Novo Nordisk - Revenue & Segment Trend Analysis

Data source: Novo Nordisk Annual Reports (20-F filings) and public financial data, 2015-2025.

Before running: upload `1_revenue_trends.csv` using the folder icon on the left sidebar.

In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt

## 1. Load data and create a SQL database

In [ ]:
df = pd.read_csv("1_revenue_trends.csv")
conn = sqlite3.connect(":memory:")
df.to_sql("revenue_data", conn, index=False, if_exists="replace")
df.head()

## 2. Annual revenue trend

In [ ]:
annual = pd.read_sql("""
    SELECT period AS year, value AS revenue_usd_million
    FROM revenue_data
    WHERE metric = 'annual_total_revenue'
    ORDER BY year
""", conn)
annual

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(annual["year"], annual["revenue_usd_million"], marker="o", linewidth=2, color="#0066B3")
plt.title("Novo Nordisk - Annual Total Revenue (2015-2025)")
plt.xlabel("Year")
plt.ylabel("Revenue (USD million)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Year-over-year growth rate

In [ ]:
yoy = pd.read_sql("""
    SELECT
        year,
        revenue_usd_million,
        ROUND(
            (revenue_usd_million - LAG(revenue_usd_million) OVER (ORDER BY year))
            * 100.0 / LAG(revenue_usd_million) OVER (ORDER BY year), 1
        ) AS yoy_growth_pct
    FROM (
        SELECT period AS year, value AS revenue_usd_million
        FROM revenue_data
        WHERE metric = 'annual_total_revenue'
    )
    ORDER BY year
""", conn)
yoy

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(yoy["year"][1:], yoy["yoy_growth_pct"][1:], color="#00A19A")
plt.title("Novo Nordisk - Year-over-Year Revenue Growth (%)")
plt.xlabel("Year")
plt.ylabel("YoY Growth (%)")
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4. Revenue by business segment

In [ ]:
segment = pd.read_sql("""
    SELECT
        period AS year,
        SUM(CASE WHEN metric = 'segment_revenue_diabetes_obesity' THEN value END) AS diabetes_obesity_dkk_m,
        SUM(CASE WHEN metric = 'segment_revenue_rare_disease' THEN value END) AS rare_disease_dkk_m
    FROM revenue_data
    WHERE metric IN ('segment_revenue_diabetes_obesity', 'segment_revenue_rare_disease')
    GROUP BY period
    ORDER BY year
""", conn)

segment["diabetes_share_pct"] = (
    segment["diabetes_obesity_dkk_m"]
    / (segment["diabetes_obesity_dkk_m"] + segment["rare_disease_dkk_m"])
    * 100
).round(1)
segment

In [ ]:
plt.figure(figsize=(9, 5))
plt.stackplot(
    segment["year"],
    segment["diabetes_obesity_dkk_m"],
    segment["rare_disease_dkk_m"],
    labels=["Diabetes & Obesity Care", "Rare Disease"],
    colors=["#0066B3", "#B0BEC5"]
)
plt.title("Novo Nordisk - Revenue by Segment (2019-2024)")
plt.xlabel("Year")
plt.ylabel("Revenue (DKK million)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()

## 5. Quarterly revenue trend

In [ ]:
quarterly = pd.read_sql("""
    SELECT period AS quarter, value AS revenue_usd_million
    FROM revenue_data
    WHERE metric = 'quarterly_total_revenue'
    ORDER BY quarter
""", conn)
quarterly

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(quarterly["quarter"], quarterly["revenue_usd_million"], marker="o", color="#E4572E")
plt.title("Novo Nordisk - Quarterly Revenue (2023 Q1 - 2026 Q1)")
plt.xlabel("Quarter")
plt.ylabel("Revenue (USD million)")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Revenue forecast (2026-2027)

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

annual["year"] = annual["year"].astype(int)
X = annual[["year"]].values
y = annual["revenue_usd_million"].values

model = LinearRegression()
model.fit(X, y)

future_years = np.array([[2026], [2027]])
forecast = model.predict(future_years)

forecast_df = pd.DataFrame({
    "year": [2026, 2027],
    "revenue_usd_million": forecast.round(0)
})
forecast_df

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(annual["year"], annual["revenue_usd_million"], marker="o", linewidth=2, color="#0066B3", label="Actual")
plt.plot(forecast_df["year"], forecast_df["revenue_usd_million"], marker="o", linewidth=2, color="#E4572E", linestyle="--", label="Forecast")
plt.title("Novo Nordisk - Revenue Forecast (2026-2027)")
plt.xlabel("Year")
plt.ylabel("Revenue (USD million)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Key findings

- Total revenue grew from ~$16.1B (2015) to ~$46.8B (2025), roughly a 3x increase.
- Growth accelerated sharply from 2022 onward, peaking at 34.6% YoY in 2023.
- Diabetes & Obesity Care's share of segment revenue rose from 84.3% (2019) to 93.6% (2024), showing increasing concentration in that segment.
- Quarterly data shows a consistent upward trend with no signs of slowdown through Q1 2026.
- A simple linear regression model (R² = 0.80) projects revenue reaching approximately $42.7B in 2026 and $45.7B in 2027 if the historical trend continues. This is a straight-line estimate and does not account for new product launches or market changes.